In [11]:
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import hashlib
import json

def now():
    return datetime.now(timezone.utc).isoformat()

def stable_id(*parts):
    return hashlib.sha256("|".join(parts).encode()).hexdigest()[:12]

@dataclass
class TraceStep:
    ts: str
    agent: str
    action: str 
    evidence: str 
    decision: str 
    authority: str  

incident = { 
    "incident_id": "INC-CLAIMS-007",
    "dataset": "insurance_claims_feed",
    "symptom": "dbt test failed: required column claim_amount missing",
    "freshness_age_minutes": 42,
    "downstream_impact": [ "claims_dashboard", "fraud_feature_table" ],
    "observed_columns": ["claim_id", "policy_id", "amount_paid", "claim_status"],
    "expected_columns": ["claim_id", "policy_id", "claim_amount", "claim_status"],
}

incident_memory = [
    {
        "pattern": "required column claim_amount missing",
        "past_fix": "map amount_paid to claim_amount only after owner approval",
        "risk": "field may not have identical business meaning",
        "verified": True,
    }
]

state = {
    "incident": incident,
    "trace": [],
    "approval": None,
    "repair_applied": False,
    "final_state": "open",
}

APPROVED_BY_HUMAN = True


def record(agent, action, evidence, decision, authority):
    state["trace"].append(TraceStep(now(), agent, action, evidence, decision, authority))

def monitor():
    evidence = f"{incident['symptom']}; freshness age {incident['freshness_age_minutes']}m"
    record("monitor", "detect_incident", evidence, "open incident", "observe_only")

def diagnose():
    missing = set(incident["expected_columns"]) - set(incident["observed_columns"])
    candiate = "amount_paid" if "amount_paid" in incident["observed_columns"] else None 
    evidence = f"missing={sorted(missing)}; candidate_replacement={candiate}"
    record("diagnoser", "compare_schema", evidence, "schema drift likely", "suggestion_only")
    return { "missing": list(missing), "candidate": candiate }

def retrieve_memory():
    match = incident_memory[0]
    evidence = f"matched pattern: {match['pattern']}; verified={match['verified']}"
    record("memory", "retrieve_similair_incident", evidence, match["past_fix"], "suggestion_only")
    return match

def plan_fix(diagnosis, memory):
    confidence  = 0.72 if diagnosis["candidate"] else 0.30
    idempotency_key = stable_id(incident["incident_id"], "schema_mapping", diagnosis.get("candidate") or "none")
    authority = "approval_required" if confidence < 0.85 else "limited_auto"
    decision = {
        "proposed_action": "map amount_paid to claim_amount in staging transform",
        "confidence": confidence,
        "authority": authority,
        "idempotency_key": idempotency_key,
        "reason": memory["risk"],
    }
    record("planner", "propise_repair", json.dumps(decision), "wait for approval", authority)
    return decision

def human_approval(plan, approved=False):
    state["approval"] = {
        "approved": approved,
        "approver": "claims_data_owner",
        "reason": "field meaining confirmed" if approved else "owner review required before shcema change",
    }
    decision = "approved" if approved else "blocked"
    record("human_approver", "review_schema_chaange", state["approval"]["reason"], decision, "approval_required")

def apply_repair(plan):
    if not state["approval"] or not state["approval"]["approved"]:
        record("fixer", "attempt_repair", "approval_missing", "do not execute", "blocked")
        return False 
    state["repaier_applied"] = True
    record("fixer", "apply_schema_mapping", plan["idempotency_key"], "repair applied in staging", "approved_execution")
    return True


def verify():
    if state["repair_applied"]:
        state["final_state"] = "verified"
        record("verifier", "run_tets", "required columns present; duplicate count 0", "close incident", "observe_only")
    else:
        state["final_state"] = "waiting_for_approval"
        record("verifier", "run_tets", "repar not applied", "incident remains open", "observe_only")


monitor()
diagnosis = diagnose()
memory = retrieve_memory()
plan = plan_fix(diagnosis, memory)
human_approval(plan, approved=APPROVED_BY_HUMAN)
apply_repair(plan)
verify()

summary = {
    "incident_id": incident["incident_id"],
    "final_state": state["final_state"],
    "approved_by_human": APPROVED_BY_HUMAN,
    "trace": [asdict(step) for step in state["trace"]]
}

pprint(summary)

output = "day-07-orchestrator-trace-approved.json" if APPROVED_BY_HUMAN else "day-07-orchestrator-trace-blocked.json"
Path(output).write_text(json.dumps(summary, indent=2))

print(f"saved {output}")




{'approved_by_human': True,
 'final_state': 'waiting_for_approval',
 'incident_id': 'INC-CLAIMS-007',
 'trace': [{'action': 'detect_incident',
            'agent': 'monitor',
            'authority': 'observe_only',
            'decision': 'open incident',
            'evidence': 'dbt test failed: required column claim_amount '
                        'missing; freshness age 42m',
            'ts': '2026-08-04T08:41:51.756149+00:00'},
           {'action': 'compare_schema',
            'agent': 'diagnoser',
            'authority': 'suggestion_only',
            'decision': 'schema drift likely',
            'evidence': "missing=['claim_amount']; "
                        'candidate_replacement=amount_paid',
            'ts': '2026-08-04T08:41:51.756248+00:00'},
           {'action': 'retrieve_similair_incident',
            'agent': 'memory',
            'authority': 'suggestion_only',
            'decision': 'map amount_paid to claim_amount only after owner '
                        